# SC3000 Project 1: SCSH, Ali & Yongjian

Contributions: <br>
We both did the lab independently and then combined the best parts of our works together.
<br>
Ali:
- Contributed validating

Yongjian:
- Made use of his generator function for Task 1
- Made use of his DPO loss function
-

Common Contributions:
- Similar approaches on general training loop
-

### Step 1: Install necesscary packages

In [2]:
# Step 0: Setup

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone repository
print("\n📥 Cloning repository...")
!git clone https://github.com/zhouyuan888888/NanoGPT-Math.git

# 3. Install dependencies
print("\n📦 Installing dependencies...")
!pip install -q torch numpy transformers datasets tiktoken wandb tqdm matplotlib gdown

# 4. Download pretrained model
print("\n⬇️ Downloading pretrained model...")
import os
os.chdir('/content/NanoGPT-Math')
!gdown 1gIZw-HAB-tHtEYCmNugwlIV7R3WsgjjZ -O ./sft/gpt.pt

# 5. Setup paths and imports
import sys
os.chdir('/content/NanoGPT-Math/dpo')
sys.path.append(os.path.abspath(".."))

import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
from tqdm import tqdm
import json
import matplotlib.pyplot as plt

# 6. Configuration
beta = 0.5
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_lr = 1e-4
epochs = 5
batch_size = 64
max_length = 64
max_new_tokens = 200
temperature = 0.8
top_k = 200

# 7. Load tokenizer
with open("../sft/meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]

def encode(s):
    return [stoi[c] for c in s]

def decode(l):
    return ''.join([itos[i] for i in l])

# 8. Verification
print("\n" + "=" * 60)
print("✓ SETUP COMPLETE!")
print("=" * 60)
print(f"✓ Device: {device}")
if not torch.cuda.is_available():
    print("⚠️  WARNING: GPU not available! Training will be slow.")
    print("   Go to: Runtime → Change runtime type → GPU")
print(f"✓ Vocabulary size: {len(stoi)}")
print(f"✓ Current directory: {os.getcwd()}")
print("✓ Ready to proceed with tasks!")
print("=" * 60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

📥 Cloning repository...
fatal: destination path 'NanoGPT-Math' already exists and is not an empty directory.

📦 Installing dependencies...

⬇️ Downloading pretrained model...
Downloading...
From (original): https://drive.google.com/uc?id=1gIZw-HAB-tHtEYCmNugwlIV7R3WsgjjZ
From (redirected): https://drive.google.com/uc?id=1gIZw-HAB-tHtEYCmNugwlIV7R3WsgjjZ&confirm=t&uuid=c14d684a-11fc-420f-918a-eeb8ac9e6d88
To: /content/NanoGPT-Math/sft/gpt.pt
100% 106M/106M [00:01<00:00, 53.8MB/s] 

✓ SETUP COMPLETE!
✓ Device: cuda
✓ Vocabulary size: 74
✓ Current directory: /content/NanoGPT-Math/dpo
✓ Ready to proceed with tasks!


In [3]:
!pip install matplotlib
!pip install torch numpy transformers datasets tiktoken wandb tqdm

### Step 2: Package imports and configuration

In [4]:
import sys
import os
sys.path.append(os.path.abspath(".."))
#os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
import random
from tqdm import tqdm
import time
import json
import matplotlib.pyplot as plt
# Configuration
beta = 0.5
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_lr = 1e-4
epochs = 5
batch_size = 64
max_length = 64
num_samples = 1
max_new_tokens = 200
temperature = 0.8
top_k = 200
# tokenizer
with open("../sft/meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

print(device)
print(stoi)


cuda
{'\n': 0, ' ': 1, "'": 2, '*': 3, '+': 4, ',': 5, '-': 6, '.': 7, '/': 8, '=': 9, '?': 10, '’': 11, '0': 12, '1': 13, '2': 14, '3': 15, '4': 16, '5': 17, '6': 18, '7': 19, '8': 20, '9': 21, 'A': 22, 'B': 23, 'C': 24, 'D': 25, 'E': 26, 'F': 27, 'G': 28, 'H': 29, 'I': 30, 'J': 31, 'K': 32, 'L': 33, 'M': 34, 'N': 35, 'O': 36, 'P': 37, 'Q': 38, 'R': 39, 'S': 40, 'T': 41, 'U': 42, 'V': 43, 'W': 44, 'X': 45, 'Y': 46, 'Z': 47, 'a': 48, 'b': 49, 'c': 50, 'd': 51, 'e': 52, 'f': 53, 'g': 54, 'h': 55, 'i': 56, 'j': 57, 'k': 58, 'l': 59, 'm': 60, 'n': 61, 'o': 62, 'p': 63, 'q': 64, 'r': 65, 's': 66, 't': 67, 'u': 68, 'v': 69, 'w': 70, 'x': 71, 'y': 72, 'z': 73}


### Step 3: Define helper functions

In [5]:
def compute_logprob(model, input_ids):
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = model(inputs, full_seq=True)

    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)

    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1) # exclude padding
    return -loss

def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

### Step 4: Load the pretrained NanoGPT model

In [6]:
ckpt = torch.load("../sft/gpt.pt", map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device)

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

### Task 1: Generate dataset

In [7]:
import random

torch.manual_seed(42)
random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

N = 250000

def make_direct():
    op = random.choice(["+", "-", "*", "/"])
    a, b = random.randint(1, 100), random.randint(1, 100)

    if op == "+":
        ans = a + b
        explanation = f"{a}+{b} equals {ans}"
    elif op == "-":
        ans = a - b
        explanation = f"{a}-{b} equals {ans}"
    elif op == "*":
        ans = a * b
        explanation = f"{a}*{b} equals {ans}"
    else:
        a = a * b
        ans = a // b
        explanation = f"{a}/{b} equals {ans}"

    q_str = f"{a}{op}{b}=?"
    return f"{q_str} The answer is {ans} because {explanation}.", f"{q_str} Sorry, I do not know."

def make_solve_x():
    op = random.choice(["+", "-", "*", "/"])

    ans, b = random.randint(1, 100), random.randint(1, 100)

    if op == "+":
        rhs = ans + b
        if random.randint(0,1) == 0:
            q_str = f"x+{b}={rhs},x=?"
        else:
            q_str = f"{b}+x={rhs},x=?"
        explanation = f"{rhs}-{b} equals to {ans}"

    elif op == "-":
        if random.randint(0,1) == 0:
            rhs = ans - b
            q_str = f"x-{b}={rhs},x=?"
            explanation = f"{rhs}+{b} equals to {ans}"
        else:
            rhs = b - ans
            q_str = f"{b}-x={rhs},x=?"
            if rhs > 0:
                explanation = f"{b}-{rhs} equals to {ans}"
            else:
                explanation = f"{b}+{-rhs} equals to {ans}"

    elif op == "*":
        rhs = ans * b
        if random.randint(0,1) == 0:
            q_str = f"x*{b}={rhs},x=?"
        else:
            q_str = f"{b}*x={rhs},x=?"
        explanation = f"{rhs}/{b} equals to {ans}"
    else:
        rhs = ans
        if random.randint(0,1) == 0:
            ans = ans * b
            rhs = ans // b
            q_str = f"x/{b}={rhs},x=?"
            explanation = f"{rhs}*{b} equals to {ans}"
        else:
            b = ans * b
            rhs = b // ans
            q_str = f"{b}/x={rhs},x=?"
            explanation = f"{b}/{rhs} equals to {ans}"
    return f"{q_str} The answer is {ans} because {explanation}.", f"{q_str} Sorry, I do not know."

neg_ans, pos_ans = [], []
json_file = "../data/pos_neg_pairs.json"


for _ in range(N//2):
    pos, neg = make_direct()
    neg_ans.append(neg)
    pos_ans.append(pos)

    pos, neg = make_solve_x()
    neg_ans.append(neg)
    pos_ans.append(pos)

data = [{"negative": n, "positive": p} for n, p in zip(neg_ans, pos_ans)]

with open(json_file, "w") as f:
    json.dump(data, f, indent=4)

print(f"Saved {len(data)} QA pairs to {json_file}")


Saved 250000 QA pairs to ../data/pos_neg_pairs.json


### Step 5: Load Data (**students are required to complete this part!**)

In [8]:
# Step 5: Load Data and Split into Train/Val/Test
import random

json_file = "../data/pos_neg_pairs.json"
with open(json_file) as f:
    lines = json.load(f)

# Shuffle data
random.seed(42)
random.shuffle(lines)

# Split: 80% train, 10% validation, 10% test
train_size = int(0.8 * len(lines))
val_size = int(0.1 * len(lines))

train_lines = lines[:train_size]
val_lines = lines[train_size:train_size + val_size]
test_lines = lines[train_size + val_size:]

print(f"Train size: {len(train_lines)}")
print(f"Validation size: {len(val_lines)}")
print(f"Test size: {len(test_lines)}")
print(f"\nExample training pair:")
print(train_lines[0])


# Define evaluation function
@torch.no_grad()
def evaluate(model, data_lines, batch_size):
    model.eval()
    total_loss = 0
    num_batches = 0

    for neg_tensor, pos_tensor in get_batches(data_lines, batch_size):
        neg_logprob = compute_logprob(model, neg_tensor)
        pos_logprob = compute_logprob(model, pos_tensor)
        loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1
        total_loss += loss.item()
        num_batches += 1

    model.train()
    return total_loss / num_batches if num_batches > 0 else float('inf')

Train size: 200000
Validation size: 25000
Test size: 25000

Example training pair:
{'negative': '84-x=26,x=? Sorry, I do not know.', 'positive': '84-x=26,x=? The answer is 58 because 84-26 equals to 58.'}


### Step 6: Build the optimizer and scheduler (**students are required to complete this part!**)

In [9]:
# Step 6: Build the optimizer and scheduler
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

epochs = 20
num_batches = len(train_lines) // batch_size
total_steps = num_batches * epochs

optim = AdamW(gpt.parameters(), lr=base_lr)
scheduler = get_linear_schedule_with_warmup(
    optim,
    num_warmup_steps=int(0.15 * total_steps),
    num_training_steps=total_steps
)

print(f"Number of training batches per epoch: {num_batches}")
print(f"Total training steps: {total_steps}")

Number of training batches per epoch: 3125
Total training steps: 62500


In [10]:
ckpt = torch.load("../sft/gpt.pt", map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
ref = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
ref.load_state_dict(state_dict)
ref.to(device)

ref.eval()



GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

### Step 7: Begin training (**students are required to complete this part!**)

In [ ]:
# Step 7: Begin training with validation
gpt.train()
best_val_loss = float('inf')
patience = 3
min_improvement = 1e-3
wait = 0

train_losses = []
val_losses = []

for epoch in range(epochs):
    epoch_loss = 0
    pbar = tqdm(get_batches(train_lines, batch_size), desc=f"Epoch {epoch+1}/{epochs}")

    for step, (neg_tensor, pos_tensor) in enumerate(pbar):
        optim.zero_grad()

        # Compute log probabilities
        neg_logprob = compute_logprob(gpt, neg_tensor)
        pos_logprob = compute_logprob(gpt, pos_tensor)

        # DPO loss without reference model
        loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1

        loss.backward()
        optim.step()
        scheduler.step()
        epoch_loss += loss.item()

        pbar.set_postfix({'train_loss': f'{loss.item():.4f}'})

    # Calculate average training loss
    avg_train_loss = epoch_loss / num_batches
    train_losses.append(avg_train_loss)

    # Evaluate on validation set
    val_loss = evaluate(gpt, val_lines, batch_size)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # Early stopping based on validation loss
    if best_val_loss - val_loss > min_improvement:
        best_val_loss = val_loss
        wait = 0
        ckpt_path = f"./dpo.pt"
        torch.save({
            "model_state_dict": gpt.state_dict(),
            "model_args": ckpt['model_args'],
        }, ckpt_path)
        print(f"✓ Saved checkpoint to {ckpt_path}")
    else:
        wait += 1
        if wait >= patience:
            print("Early stopping!")
            break

# Plot training curves
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.show()

Epoch 1/20: 3125it [02:16, 22.95it/s, train_loss=0.0826]


Epoch 1 | Train Loss: 0.5018 | Val Loss: 0.0777
✓ Saved checkpoint to ./dpo.pt


Epoch 2/20: 3125it [02:13, 23.38it/s, train_loss=0.0516]


Epoch 2 | Train Loss: 0.0681 | Val Loss: 0.0460
✓ Saved checkpoint to ./dpo.pt


Epoch 3/20: 3125it [02:14, 23.28it/s, train_loss=0.0298]


Epoch 3 | Train Loss: 0.0342 | Val Loss: 0.0283
✓ Saved checkpoint to ./dpo.pt


Epoch 4/20: 3125it [02:13, 23.41it/s, train_loss=0.0263]


Epoch 4 | Train Loss: 0.0278 | Val Loss: 0.0251
✓ Saved checkpoint to ./dpo.pt


Epoch 5/20: 1555it [01:06, 23.59it/s, train_loss=0.0251]

### Step 8: Begin testing (**students are required to complete this part!**)

In [12]:
# Step 8: Final Evaluation on Test Set

# Load the best model
ckpt_path = "./dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).to(device)
state_dict = checkpoint['model_state_dict']
gpt.load_state_dict(state_dict)

# Evaluate on test set
test_loss = evaluate(gpt, test_lines, batch_size)
print(f"Test Loss: {test_loss:.4f}\n")

# Test on sample questions
gpt.eval()
test_set = ["17+19=?", "3*17=?", "72/4=?", "72-x=34,x=?", "x*11=44,x=?", "3*17=?", "72/4=?", "72-x=34,x=?"]

print("=" * 60)
print("SAMPLE TEST OUTPUTS:")
print("=" * 60)

with torch.no_grad():
    for prompt in test_set:
        prompt_ids = encode(prompt)
        x = torch.tensor(prompt_ids, dtype=torch.long, device=device)[None, ...]
        y = gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        output = decode(y[0][0].tolist())
        print(f"\n{output}")
        print("-" * 60)

NameError: name 'evaluate' is not defined

# PART 2: Further Fine-tuning on More Stringent Dataset

In this section, we perform additional fine-tuning using a more challenging dataset where:
- **Positive examples**: Correct answers WITH full explanations
- **Negative examples**: Wrong answers WITHOUT explanations

This helps the model learn to distinguish between correct reasoning and mere guessing.

In [ ]:
# Part 2 - Step 1: Define mutation function for creating wrong answers

import random

def create_mutated_answer(correct_ans):
    """
    Takes a correct integer answer and returns a new string-based answer that is
    numerically incorrect.

    This introduces several types of errors:
    - Mutating a single digit (e.g., 36 -> 39)
    - Adding a digit (e.g., 36 -> 364)
    - Being off by a small amount (e.g., 36 -> 37)
    - Removing a digit (e.g., 36 -> 3)
    """
    correct_ans_str = str(correct_ans)

    # Choose a mutation strategy
    mutation = random.choice(['digit', 'digit', 'off_by', 'off_by', 'add', 'remove'])

    # Strategy 1: Mutate a single digit
    if mutation == 'digit':
        if correct_ans == 0:
            return str(random.randint(1, 9))

        idx = random.randint(0, len(correct_ans_str) - 1)
        new_digit = str(random.randint(0, 9))

        while new_digit == correct_ans_str[idx]:
            new_digit = str(random.randint(0, 9))

        wrong_ans_str = correct_ans_str[:idx] + new_digit + correct_ans_str[idx+1:]

    # Strategy 2: Add a random digit to the end
    elif mutation == 'add':
        wrong_ans_str = correct_ans_str + str(random.randint(0, 9))

    # Strategy 3: Off by a small amount
    elif mutation == 'off_by':
        delta = random.choice([-3, -2, -1, 1, 2, 3])
        new_ans = correct_ans + delta
        if new_ans < 0:
            new_ans = correct_ans + abs(delta)
        wrong_ans_str = str(new_ans)

    # Strategy 4: Remove the last digit
    else:  # 'remove'
        if len(correct_ans_str) > 1:
            wrong_ans_str = correct_ans_str[:-1]
        else:
            wrong_ans_str = str((correct_ans + random.randint(1, 9)) % 10)

    return wrong_ans_str

In [ ]:
# Part 2 - Step 2a: Generate direct arithmetic pairs

def generate_direct_pair_v3():
    """
    Generate direct arithmetic question pairs (e.g., "17+19=?")
    Positive: Full answer with explanation
    Negative: Wrong answer WITHOUT explanation
    """
    op = random.choice(["+", "-", "*", "/"])
    a, b = random.randint(1, 100), random.randint(1, 100)

    if op == "+":
        ans = a + b
        explanation = f"{a}+{b} equals {ans}"
    elif op == "-":
        ans = a - b
        explanation = f"{a}-{b} equals {ans}"
    elif op == "*":
        ans = a * b
        explanation = f"{a}*{b} equals {ans}"
    else:
        a = a * b
        ans = a // b
        explanation = f"{a}/{b} equals {ans}"

    q_str = f"{a}{op}{b}=?"

    # POSITIVE: Correct answer with full explanation
    qa_pos = f"{q_str} The answer is {ans} because {explanation}."

    # NEGATIVE: Wrong answer without proper explanation
    wrong_ans_str = create_mutated_answer(ans)

    # Choose a negative format
    neg_format = random.choice(['no_explain', 'incomplete', 'uncertain', 'just_wrong'])

    if neg_format == 'no_explain':
        qa_neg = f"{q_str} The answer is {wrong_ans_str}."
    elif neg_format == 'incomplete':
        qa_neg = f"{q_str} The answer is {wrong_ans_str}"
    elif neg_format == 'uncertain':
        qa_neg = f"{q_str} I think it's {wrong_ans_str}."
    else:  # 'just_wrong'
        qa_neg = f"{q_str} {wrong_ans_str}"

    return {"positive": qa_pos, "negative": qa_neg}

print("✓ Direct arithmetic pair generator defined")

In [ ]:
# Part 2 - Step 2b: Generate solve-for-x pairs

def generate_solve_x_pair_v3():
    """
    Generate solve-for-x question pairs (e.g., "x+5=12,x=?")
    Positive: Full answer with explanation
    Negative: Wrong answer WITHOUT explanation
    """
    op = random.choice(["+", "-", "*", "/"])
    ans, b = random.randint(1, 100), random.randint(1, 100)

    q_str = ""
    explanation = ""

    if op == "+":
        rhs = ans + b
        if random.randint(0, 1) == 0:
            q_str = f"x+{b}={rhs},x=?"
        else:
            q_str = f"{b}+x={rhs},x=?"
        explanation = f"{rhs}-{b} equals to {ans}"

    elif op == "-":
        if random.randint(0, 1) == 0:
            rhs = ans - b
            q_str = f"x-{b}={rhs},x=?"
            explanation = f"{rhs}+{b} equals to {ans}"
        else:
            rhs = b - ans
            q_str = f"{b}-x={rhs},x=?"
            if rhs > 0:
                explanation = f"{b}-{rhs} equals to {ans}"
            else:
                explanation = f"{b}+{-rhs} equals to {ans}"

    elif op == "*":
        rhs = ans * b
        if random.randint(0, 1) == 0:
            q_str = f"x*{b}={rhs},x=?"
        else:
            q_str = f"{b}*x={rhs},x=?"
        explanation = f"{rhs}/{b} equals to {ans}"

    else:  # "/"
        if random.randint(0, 1) == 0:
            x_val = ans * b
            rhs = x_val // b
            q_str = f"x/{b}={rhs},x=?"
            explanation = f"{rhs}*{b} equals to {x_val}"
            ans = x_val
        else:
            dividend = ans * b
            rhs = dividend // ans
            q_str = f"{dividend}/x={rhs},x=?"
            explanation = f"{dividend}/{rhs} equals to {ans}"

    # POSITIVE: Correct answer with full explanation
    qa_pos = f"{q_str} The answer is {ans} because {explanation}."

    # NEGATIVE: Wrong answer without proper explanation
    wrong_ans_str = create_mutated_answer(ans)

    # Choose a negative format
    neg_format = random.choice(['no_explain', 'incomplete', 'uncertain', 'just_wrong'])

    if neg_format == 'no_explain':
        qa_neg = f"{q_str} The answer is {wrong_ans_str}."
    elif neg_format == 'incomplete':
        qa_neg = f"{q_str} The answer is {wrong_ans_str}"
    elif neg_format == 'uncertain':
        qa_neg = f"{q_str} I think x is {wrong_ans_str}."
    else:  # 'just_wrong'
        qa_neg = f"{q_str} {wrong_ans_str}"

    return {"positive": qa_pos, "negative": qa_neg}

print("✓ Solve-for-x pair generator defined")

In [ ]:
# Part 2 - Step 3: Generate V3 dataset

N_v3 = 50000  # Generate 50k samples for Part 2
json_file_v3 = "../data/pos_neg_pairs_v3.json"
data_v3 = []

print(f"Generating {N_v3} improved pairs...")
print("Positive: Correct answer with full explanation")
print("Negative: Wrong answer WITHOUT explanation")
print()

for i in range(N_v3 // 2):
    data_v3.append(generate_direct_pair_v3())
    data_v3.append(generate_solve_x_pair_v3())

    if (i + 1) % 5000 == 0:
        print(f"Generated {len(data_v3)} pairs...")

# Save the new dataset
with open(json_file_v3, "w") as f:
    json.dump(data_v3, f, indent=4)

print(f"\n✓ Successfully generated and saved {len(data_v3)} V3 pairs to {json_file_v3}")
print("\n--- Example V3 Pairs ---")
for i in range(3):
    print(f"\nPair {i+1}:")
    print(json.dumps(data_v3[i], indent=2))

In [ ]:
# Part 2 - Step 4: Split V3 data into train/val/test

# Load the V3 dataset
json_file_v3 = "../data/pos_neg_pairs_v3.json"
with open(json_file_v3) as f:
    lines_v3 = json.load(f)

print(f"Loaded {len(lines_v3)} V3 pairs")

# Set seed for reproducibility
random.seed(42)

# Shuffle the V3 data
random.shuffle(lines_v3)

# Split ratios
train_ratio = 0.8
val_ratio = 0.1
test_ratio = 0.1

# Calculate split indices
total_size = len(lines_v3)
train_size = int(total_size * train_ratio)
val_size = int(total_size * val_ratio)

# Split the V3 data
train_lines_v3 = lines_v3[:train_size]
val_lines_v3 = lines_v3[train_size:train_size + val_size]
test_lines_v3 = lines_v3[train_size + val_size:]

print(f"\nTotal V3 data: {total_size}")
print(f"Train: {len(train_lines_v3)}, Val: {len(val_lines_v3)}, Test: {len(test_lines_v3)}")

print("\n--- Example from each split ---")
print("\nTrain example:")
print(json.dumps(train_lines_v3[0], indent=2))
print("\nValidation example:")
print(json.dumps(val_lines_v3[0], indent=2))
print("\nTest example:")
print(json.dumps(test_lines_v3[0], indent=2))

In [ ]:
# Part 2 - Step 5: Load the Part 1 fine-tuned model as starting point

# Load the fine-tuned model from Part 1
ckpt_path_part1 = "./dpo.pt"
print(f"Loading Part 1 model from {ckpt_path_part1}...")

checkpoint = torch.load(ckpt_path_part1, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).to(device)

state_dict = checkpoint.get('model_state_dict', checkpoint.get('model'))
gpt.load_state_dict(state_dict)

print("✓ Part 1 model loaded successfully")

# Update configuration for Part 2
epochs_v3 = 15  # Can adjust this
num_batches_v3 = len(train_lines_v3) // batch_size
total_steps_v3 = num_batches_v3 * epochs_v3

print(f"\nPart 2 Training Configuration:")
print(f"  Epochs: {epochs_v3}")
print(f"  Batches per epoch: {num_batches_v3}")
print(f"  Total steps: {total_steps_v3}")

# Reinitialize optimizer and scheduler for Part 2
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

optim_v3 = AdamW(gpt.parameters(), lr=base_lr)
scheduler_v3 = get_linear_schedule_with_warmup(
    optim_v3,
    num_warmup_steps=int(0.15 * total_steps_v3),
    num_training_steps=total_steps_v3
)

print("\n✓ Optimizer and scheduler initialized for Part 2")

In [ ]:
# Part 2 - Step 6: Define evaluation function for V3 data

def evaluate_v3(model, eval_lines, batch_size):
    """
    Evaluate the model on validation/test data
    """
    model.eval()
    total_loss = 0
    num_batches_eval = 0

    with torch.no_grad():
        for neg_tensor, pos_tensor in get_batches(eval_lines, batch_size):
            neg_logprob = compute_logprob(model, neg_tensor)
            pos_logprob = compute_logprob(model, pos_tensor)

            loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1
            total_loss += loss.item()
            num_batches_eval += 1

    model.train()
    return total_loss / num_batches_eval if num_batches_eval > 0 else float('inf')

print("✓ Evaluation function defined")

In [ ]:
# Part 2 - Step 7: Begin training on V3 dataset

gpt.train()
best_val_loss_v3 = float('inf')
patience_v3 = 3
min_improvement_v3 = 1e-3
wait_v3 = 0

train_losses_v3 = []
val_losses_v3 = []

print("=" * 80)
print("PART 2 TRAINING - V3 Dataset (More Stringent Negatives)")
print("=" * 80)

for epoch in range(epochs_v3):
    epoch_loss = 0
    pbar = tqdm(get_batches(train_lines_v3, batch_size), desc=f"Epoch {epoch+1}/{epochs_v3}")

    for step, (neg_tensor, pos_tensor) in enumerate(pbar):
        optim_v3.zero_grad()

        # Compute log probabilities
        neg_logprob = compute_logprob(gpt, neg_tensor)
        pos_logprob = compute_logprob(gpt, pos_tensor)

        # DPO loss
        loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1

        loss.backward()
        optim_v3.step()
        scheduler_v3.step()
        epoch_loss += loss.item()

        pbar.set_postfix({'train_loss': f'{loss.item():.4f}'})

    # Calculate average training loss
    avg_train_loss = epoch_loss / num_batches_v3
    train_losses_v3.append(avg_train_loss)

    # Evaluate on validation set
    val_loss = evaluate_v3(gpt, val_lines_v3, batch_size)
    val_losses_v3.append(val_loss)

    print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # Early stopping based on validation loss
    if best_val_loss_v3 - val_loss > min_improvement_v3:
        best_val_loss_v3 = val_loss
        wait_v3 = 0
        ckpt_path_v3 = "./dpo_v3.pt"
        torch.save({
            "model_state_dict": gpt.state_dict(),
            "model_args": checkpoint['model_args'],
        }, ckpt_path_v3)
        print(f"✓ Saved Part 2 checkpoint to {ckpt_path_v3}")
    else:
        wait_v3 += 1
        if wait_v3 >= patience_v3:
            print("Early stopping!")
            break

# Plot training curves for Part 2
plt.figure(figsize=(10, 5))
plt.plot(train_losses_v3, label='Train Loss (V3)', marker='o')
plt.plot(val_losses_v3, label='Validation Loss (V3)', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Part 2: Training and Validation Loss (V3 Dataset)')
plt.legend()
plt.grid(True)
plt.show()

print("\n✓ Part 2 training complete!")

In [ ]:
# Part 2 - Step 8: Test the Part 2 fine-tuned model

# Load the best Part 2 model
ckpt_path_v3 = "./dpo_v3.pt"
checkpoint_v3 = torch.load(ckpt_path_v3, map_location=device)
gptconf_v3 = GPTConfig(**checkpoint_v3['model_args'])
gpt_v3 = GPT(gptconf_v3).to(device)

state_dict_v3 = checkpoint_v3.get('model_state_dict', checkpoint_v3.get('model'))
gpt_v3.load_state_dict(state_dict_v3)
gpt_v3.eval()

print("✓ Part 2 model loaded")
print()

# Test on sample questions
test_questions = [
    "17+19=?",
    "3*17=?",
    "72/4=?",
    "72-x=34,x=?",
    "x*11=44,x=?",
    "100-45=?",
    "x+25=60,x=?",
    "8*9=?",
    "22/2=?",
    "99/11=?",
    "27/9=?"
]

print("=" * 80)
print("PART 2 - SAMPLE TEST OUTPUTS")
print("=" * 80)

with torch.no_grad():
    for prompt in test_questions:
        prompt_ids = encode(prompt)
        x = torch.tensor(prompt_ids, dtype=torch.long, device=device)[None, ...]
        y, _ = gpt_v3.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        output = decode(y[0].tolist())
        print(f"\n{output}")
        print("-" * 80)

In [ ]:
# Part 2 - Step 9: Automated evaluation on test set

import re

def extract_answer(text):
    """Extract numerical answer from model output using regex."""
    # Try to find "answer is <number>"
    match = re.search(r'answer is\s+(-?\d+)', text, re.IGNORECASE)
    if match:
        return int(match.group(1))

    # Try to find "x = <number>" or "x=<number>"
    match = re.search(r'x\s*=\s*(-?\d+)', text, re.IGNORECASE)
    if match:
        return int(match.group(1))

    # Try to find "equals <number>"
    match = re.search(r'equals\s+(-?\d+)', text, re.IGNORECASE)
    if match:
        return int(match.group(1))

    return None

def compute_ground_truth(question):
    """Compute the ground truth answer for a given question."""
    try:
        # Handle direct arithmetic: "17+19=?"
        match = re.match(r'(\d+)([\+\-\*\/])(\d+)=\?', question)
        if match:
            a, op, b = int(match.group(1)), match.group(2), int(match.group(3))
            if op == '+': return a + b
            elif op == '-': return a - b
            elif op == '*': return a * b
            elif op == '/': return a // b

        # Handle "x+b=c,x=?" or "b+x=c,x=?"
        match = re.match(r'x\+(\d+)=(\d+),x=\?', question)
        if match:
            b, c = int(match.group(1)), int(match.group(2))
            return c - b
        match = re.match(r'(\d+)\+x=(\d+),x=\?', question)
        if match:
            b, c = int(match.group(1)), int(match.group(2))
            return c - b

        # Handle "x-b=c,x=?"
        match = re.match(r'x-(\d+)=(-?\d+),x=\?', question)
        if match:
            b, c = int(match.group(1)), int(match.group(2))
            return c + b

        # Handle "b-x=c,x=?"
        match = re.match(r'(\d+)-x=(-?\d+),x=\?', question)
        if match:
            b, c = int(match.group(1)), int(match.group(2))
            return b - c

        # Handle "x*b=c,x=?" or "b*x=c,x=?"
        match = re.match(r'x\*(\d+)=(\d+),x=\?', question)
        if match:
            b, c = int(match.group(1)), int(match.group(2))
            return c // b
        match = re.match(r'(\d+)\*x=(\d+),x=\?', question)
        if match:
            b, c = int(match.group(1)), int(match.group(2))
            return c // b

        # Handle "x/b=c,x=?"
        match = re.match(r'x\/(\d+)=(\d+),x=\?', question)
        if match:
            b, c = int(match.group(1)), int(match.group(2))
            return c * b

        # Handle "b/x=c,x=?"
        match = re.match(r'(\d+)\/x=(\d+),x=\?', question)
        if match:
            b, c = int(match.group(1)), int(match.group(2))
            return b // c
    except:
        pass

    return None

# Evaluate on V3 test set
correct = 0
total = 0
errors = []

print("=" * 80)
print("PART 2 - AUTOMATED TEST SET EVALUATION")
print("=" * 80)

with torch.no_grad():
    # Sample from test set (use first 100 for faster evaluation)
    test_sample = random.sample(test_lines_v3, min(100, len(test_lines_v3)))

    for item in tqdm(test_sample, desc="Evaluating Part 2"):
        # Extract question from positive example
        positive = item['positive']
        question = positive.split('The answer')[0].strip()

        # Generate model response
        prompt_ids = encode(question)
        x = torch.tensor(prompt_ids, dtype=torch.long, device=device)[None, ...]
        y, _ = gpt_v3.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        response = decode(y[0].tolist())

        # Extract answers
        predicted_answer = extract_answer(response)
        ground_truth = compute_ground_truth(question)

        if ground_truth is not None and predicted_answer is not None:
            total += 1
            if predicted_answer == ground_truth:
                correct += 1
            else:
                errors.append({
                    'question': question,
                    'predicted': predicted_answer,
                    'ground_truth': ground_truth,
                    'response': response
                })

print(f"\n{'=' * 80}")
print(f"PART 2 RESULTS:")
print(f"{'=' * 80}")
print(f"Total evaluated: {total}")
print(f"Correct: {correct}")
print(f"Accuracy: {100 * correct / total:.2f}%" if total > 0 else "N/A")
print(f"{'=' * 80}\n")

# Show some example errors
if errors:
    print("\nExample errors (first 5):")
    print("-" * 80)
    for i, err in enumerate(errors[:5]):
        print(f"\n{i+1}. Question: {err['question']}")
        print(f"   Ground truth: {err['ground_truth']}")
        print(f"   Predicted: {err['predicted']}")
        print(f"   Full response: {err['response'][:100]}...")